# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed and up to date
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Cite as:", metadata.citeAs)
print("License:", metadata.license)
print("Date Published:", metadata.datePublished)

## 2. Data Overview
Review available record sets, fields, and their IDs.

**All references use `@id` fields as required.**

In [ ]:
# List all record sets from the dataset schema
record_sets = dataset.record_sets
print(f"Available RecordSets ({len(record_sets)}):")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.dataType})")
    print()

In [ ]:
# Show a sample of records from the first record set
if record_sets:
    first_record_set_id = record_sets[0].id
    sample_records = dataset.records(record_set=first_record_set_id)
    for i, rec in enumerate(sample_records):
        print(f"Sample #{i+1}: {rec}")
        if i >= 2:
            break
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s listed above.

### *Example extraction: Each record set loaded by its `@id`*

In [ ]:
# Extract data from each record set by its @id
dataframes = {}

record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @id: {record_set_id}, shape: {df.shape}")

# Print columns of first record set
if record_set_ids:
    print("Columns in first RecordSet:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    dataframes[record_set_ids[0]].head()
else:
    print("No dataframes to show.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All fields referenced by their `@id`s.**

In [ ]:
# Identify numeric fields by their @id in the first record set
first_rs = record_sets[0] if record_sets else None

numeric_fields = [f for f in first_rs.fields if f.dataType in ['Integer', 'Float', 'Number']]
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    print(f"Numeric field chosen: {numeric_fields[0].name} (@id: {numeric_field_id})")
else:
    numeric_field_id = None
    print("No numeric fields found.")

df = dataframes[first_rs.id] if first_rs else pd.DataFrame()

# Filtering: Keep rows where the numeric field > threshold
if numeric_field_id and not df.empty:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by another field (e.g. categorical with @id)
    group_fields = [f for f in first_rs.fields if f.dataType == 'Text' or f.dataType == 'Category']
    if group_fields:
        group_field_id = group_fields[0].id
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No numeric field found in dataframe or dataframe is empty.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field in the first record set
if numeric_field_id and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot: numeric vs group field
    if group_fields:
        group_field_id = group_fields[0].id
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration:

- The dataset contains clinical and molecular details of second primary colorectal cancer in cancer survivors, with variables such as MSI status, anatomical distribution, and comorbidities.
- Record sets, fields, and columns are identified by their unique `@id`s for robust data handling.
- Applied filtering, normalization, and basic grouping by attributes provide insight into data distributions.
- Visualized numeric and categorical field relationships for exploratory analysis.

This workflow can be extended for further clinical or machine learning analysis, using `mlcroissant` and referenced `@id`s as required by the FAIR^2 Croissant schema.